# Inherent Sample Profiling — PER METRIK

Versi lanjutan: seluruh audit karakteristik bawaan (korelasi ranking, ICC, klasifikasi profil) dijalankan
**terpisah untuk tiap metrik** — `coherence`, `relevance`, `fluency`, `consistency` — bukan hanya pada
komposit `_ling`.

Pertanyaan yang dijawab:
- Apakah sifat bawaan kesulitan itu **konsisten antar metrik**? (dialog sulit di fluency juga sulit di coherence?)
- Atau **metrik-spesifik**? (ada dialog yang selalu rendah HANYA di fluency, normal di lain)
- Metrik mana yang paling "ditentukan dialog" (ICC tertinggi) vs paling "ditentukan config" (ICC rendah)?

> `unieval.json` tiap config = list per-instance, urutan dialog sama (paired).

## 1. Loader & muat data

In [1]:
import json
from pathlib import Path
import numpy as np, pandas as pd

def load_json(path):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"Tidak ditemukan: {p}")
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

METHOD = "method4"

def load_per_instance(folder):
    """Muat unieval.json (list per-instance) -> DataFrame per dialog."""
    data = load_json(f"{folder}/unieval.json")
    align_score = load_json(f"{folder}/alignscore.json")

    assert len(data) == len(align_score), (
        f"Length mismatch: data={len(data)}, align_score={len(align_score)}"
    )

    rows = [
        {**row, "alignscore": score}
        for row, score in zip(data, align_score)
    ]

    return pd.DataFrame(rows)
    
CONFIGS = {
    "margin_part": f"{METHOD}_outputs/bart_margin_part",
    "margin_full": f"{METHOD}_outputs/bart_margin_full",
    "finetune":    f"{METHOD}_outputs/bart_finetune",
}
LING = ["coherence", "relevance", "fluency", "consistency", "alignscore"]

per = {name: load_per_instance(folder) for name, folder in CONFIGS.items()}
n = min(len(d) for d in per.values())
for k in per:
    per[k] = per[k].iloc[:n].reset_index(drop=True)
    per[k]["_ling"] = per[k][LING].mean(axis=1)
print("Jumlah dialog (paired):", n, "| config:", list(per))

# Bangun satu matriks skor per metrik: M[metrik] -> DataFrame (baris=dialog, kolom=config)
METRICS = LING + ["_ling"]
M = {m: pd.DataFrame({k: per[k][m].values for k in per}) for m in METRICS}
for m in METRICS:
    M[m].index.name = "dialog_id"

Jumlah dialog (paired): 499 | config: ['margin_part', 'margin_full', 'finetune']


## 2. Fungsi analisis (dipakai ulang per metrik)

In [2]:
from scipy.stats import spearmanr, kendalltau
from itertools import combinations

def icc_2_1(Mat):
    """ICC(2,1) two-way random, single rater. baris=dialog, kolom=config."""
    Mat = np.asarray(Mat, float)
    n_s, k = Mat.shape
    grand = Mat.mean(); row_m = Mat.mean(axis=1); col_m = Mat.mean(axis=0)
    SS_t = ((Mat - grand) ** 2).sum()
    SS_r = k * ((row_m - grand) ** 2).sum()
    SS_c = n_s * ((col_m - grand) ** 2).sum()
    SS_e = SS_t - SS_r - SS_c
    MS_r = SS_r / (n_s - 1); MS_c = SS_c / (k - 1); MS_e = SS_e / ((n_s - 1) * (k - 1))
    return (MS_r - MS_e) / (MS_r + (k - 1) * MS_e + k * (MS_c - MS_e) / n_s)

def mean_spearman(S):
    vals = [spearmanr(S[a], S[b])[0] for a, b in combinations(S.columns, 2)]
    return float(np.mean(vals)), vals

HI, LO, SWING = 0.80, 0.10, 0.50  # ambang percentile

def classify_profiles(S):
    """Kembalikan Series profil per dialog untuk satu metrik (matriks S: dialog x config)."""
    P = S.rank(pct=True)
    pmin, pmax = P.min(axis=1), P.max(axis=1)
    prng = pmax - pmin
    out = pd.Series("neutral", index=S.index)
    out[prng >= SWING] = "swinger"
    out[(prng < SWING) & (pmin >= HI)] = "always_high"
    out[(prng < SWING) & (pmax <= LO)] = "always_low"
    return out, P

## 3. Korelasi ranking & ICC — untuk SETIAP metrik

Satu baris ringkasan per metrik. ICC tinggi + ρ tinggi = pada metrik itu, kesulitan dialog adalah sifat bawaan.

In [3]:
rows = []
for m in METRICS:
    S = M[m]
    icc = icc_2_1(S.values)
    mrho, _ = mean_spearman(S)
    rows.append({"metric": m, "ICC": round(icc, 3), "mean_Spearman": round(mrho, 3),
                 "mean_score": round(S.values.mean(), 4)})
summary = pd.DataFrame(rows).set_index("metric")

print("="*64)
print("RINGKASAN PER METRIK")
print("="*64)
print(summary.to_string())
print("""
Baca:
- ICC tinggi (>0.75) + ρ tinggi  -> sifat bawaan kuat pada metrik ini; config hampir tak mengubah ranking.
- ICC rendah (<0.5)              -> pada metrik ini, config-lah penentu; bukan kesulitan bawaan dialog.
- Bandingkan antar metrik: metrik dengan ICC paling rendah adalah tempat eksperimen Anda paling 'bekerja'.""")

RINGKASAN PER METRIK
               ICC  mean_Spearman  mean_score
metric                                       
coherence    0.165          0.326      0.8980
relevance    0.217          0.385      0.8800
fluency      0.382          0.366      0.8362
consistency  0.362          0.413      0.8977
alignscore   0.431          0.440      0.7869
_ling        0.285          0.360      0.8598

Baca:
- ICC tinggi (>0.75) + ρ tinggi  -> sifat bawaan kuat pada metrik ini; config hampir tak mengubah ranking.
- ICC rendah (<0.5)              -> pada metrik ini, config-lah penentu; bukan kesulitan bawaan dialog.
- Bandingkan antar metrik: metrik dengan ICC paling rendah adalah tempat eksperimen Anda paling 'bekerja'.


## 4. Apakah kesulitan konsisten ANTAR metrik?

Untuk tiap config, hitung korelasi Spearman antar metrik atas skor per-dialog, lalu rata-ratakan lintas config.
ρ antar-metrik tinggi = dialog yang sulit di satu metrik juga sulit di metrik lain (kesulitan umum, bukan
metrik-spesifik).

In [4]:
# rata-rata skor tiap metrik lintas config (per dialog) lalu korelasikan antar metrik
avg_by_metric = pd.DataFrame({m: M[m].mean(axis=1).values for m in LING})
cm = avg_by_metric.corr(method="spearman").round(3)
print("="*64)
print("KORELASI SPEARMAN ANTAR METRIK (skor per-dialog, dirata-rata lintas config)")
print("="*64)
print(cm.to_string())
tri = cm.values[np.triu_indices_from(cm.values, k=1)]
print(f"\nRata-rata ρ antar-metrik = {tri.mean():+.3f}")
print("  tinggi -> kesulitan bersifat UMUM (dialog sulit di semua metrik).")
print("  rendah -> kesulitan METRIK-SPESIFIK (dialog bisa sulit di fluency tapi baik di relevance).")

KORELASI SPEARMAN ANTAR METRIK (skor per-dialog, dirata-rata lintas config)
             coherence  relevance  fluency  consistency  alignscore
coherence        1.000      0.755    0.406        0.659       0.100
relevance        0.755      1.000    0.434        0.606       0.140
fluency          0.406      0.434    1.000        0.494       0.099
consistency      0.659      0.606    0.494        1.000       0.250
alignscore       0.100      0.140    0.099        0.250       1.000

Rata-rata ρ antar-metrik = +0.394
  tinggi -> kesulitan bersifat UMUM (dialog sulit di semua metrik).
  rendah -> kesulitan METRIK-SPESIFIK (dialog bisa sulit di fluency tapi baik di relevance).


## 5. Klasifikasi profil per dialog, untuk SETIAP metrik

Hasilnya satu kolom profil per metrik, ditambah kolom komposit. Dari sini terlihat dialog yang `always_low`
hanya di metrik tertentu.

In [5]:
profiles = pd.DataFrame(index=range(n)); profiles.index.name = "dialog_id"
for m in METRICS:
    prof_m, _ = classify_profiles(M[m])
    profiles[f"prof_{m}"] = prof_m.values
    profiles[f"mean_{m}"] = M[m].mean(axis=1).round(4).values

print("="*64)
print("DISTRIBUSI PROFIL per metrik")
print("="*64)
dist = pd.DataFrame({m: profiles[f"prof_{m}"].value_counts() for m in METRICS}).fillna(0).astype(int)
print(dist.to_string())

DISTRIBUSI PROFIL per metrik
             coherence  relevance  fluency  consistency  alignscore  _ling
neutral            322        343      324          347         345    335
swinger            151        129      152          117         118    139
always_high         24         21       16           29          29     21
always_low           2          6        7            6           7      4


## 6. Dialog yang selalu rendah — di berapa metrik?

Menghitung, per dialog, di berapa metrik ia tergolong `always_low`. `n_low = 4` berarti dialog itu sulit
bawaan di SEMUA metrik linguistik (kandidat terkuat). `n_low = 1` berarti masalahnya metrik-spesifik.

In [6]:
low_flags  = pd.DataFrame({m: (profiles[f"prof_{m}"] == "always_low")  for m in LING})
high_flags = pd.DataFrame({m: (profiles[f"prof_{m}"] == "always_high") for m in LING})
swing_flags= pd.DataFrame({m: (profiles[f"prof_{m}"] == "swinger")     for m in LING})

agg = pd.DataFrame(index=profiles.index)
agg["n_low_metrics"]   = low_flags.sum(axis=1)
agg["low_in"]          = low_flags.apply(lambda r: ",".join([m for m in LING if r[m]]), axis=1)
agg["n_high_metrics"]  = high_flags.sum(axis=1)
agg["n_swing_metrics"] = swing_flags.sum(axis=1)
agg["swing_in"]        = swing_flags.apply(lambda r: ",".join([m for m in LING if r[m]]), axis=1)
agg["mean_ling"]       = M["_ling"].mean(axis=1).round(4).values

print("="*64)
print("ALWAYS_LOW di SEMUA 4 metrik (sulit bawaan universal)")
print("="*64)
universal_low = agg[agg["n_low_metrics"] == len(LING)].sort_values("mean_ling")
print(universal_low.to_string() if len(universal_low) else "(tidak ada)")

print("\n" + "="*64)
print("ALWAYS_LOW metrik-spesifik (hanya 1-3 metrik) — 20 teratas")
print("="*64)
specific_low = agg[(agg["n_low_metrics"] >= 1) & (agg["n_low_metrics"] < len(LING))].sort_values("n_low_metrics", ascending=False)
print(specific_low.head(20).to_string() if len(specific_low) else "(tidak ada)")

ALWAYS_LOW di SEMUA 4 metrik (sulit bawaan universal)
(tidak ada)

ALWAYS_LOW metrik-spesifik (hanya 1-3 metrik) — 20 teratas
           n_low_metrics                  low_in  n_high_metrics  n_swing_metrics           swing_in  mean_ling
dialog_id                                                                                                      
60                     2  consistency,alignscore               0                0                        0.6990
300                    2       relevance,fluency               0                0                        0.6387
39                     1             consistency               0                1         alignscore     0.7591
247                    1             consistency               0                1         alignscore     0.7465
408                    1              alignscore               0                0                        0.8381
395                    1               coherence               0                1         

## 7. Swinger per metrik — arah perubahan

Untuk tiap metrik, dialog swinger beserta config terbaik/terburuknya. Ini menjawab eksplisit "rendah di
Eksperimen A, tinggi di Eksperimen B" pada metrik yang dimaksud.

In [7]:
for m in LING:
    S = M[m]
    prof_m, P = classify_profiles(S)
    idx = prof_m[prof_m == "swinger"].index
    print("="*64)
    print(f"SWINGER pada metrik: {m}  ({len(idx)} dialog)")
    print("="*64)
    if len(idx) == 0:
        print("(tidak ada pada ambang ini)\n"); continue
    d = pd.DataFrame(index=idx)
    d["best_config"]  = S.loc[idx].idxmax(axis=1)
    d["worst_config"] = S.loc[idx].idxmin(axis=1)
    d["best"]  = S.loc[idx].max(axis=1).round(4)
    d["worst"] = S.loc[idx].min(axis=1).round(4)
    d["gap"]   = (d["best"] - d["worst"]).round(4)
    print(d.sort_values("gap", ascending=False).head(10).to_string())
    print()

SWINGER pada metrik: coherence  (151 dialog)
           best_config worst_config    best   worst     gap
dialog_id                                                  
135           finetune  margin_part  0.9682  0.0350  0.9332
86         margin_part     finetune  0.9690  0.0378  0.9312
0             finetune  margin_full  0.9820  0.0562  0.9258
406        margin_part     finetune  0.9717  0.0587  0.9130
13         margin_full  margin_part  0.9689  0.0630  0.9059
240        margin_part     finetune  0.9643  0.0592  0.9051
426           finetune  margin_full  0.9649  0.0715  0.8934
339           finetune  margin_full  0.9823  0.0944  0.8879
357        margin_part     finetune  0.9691  0.0905  0.8786
283        margin_part     finetune  0.9654  0.1062  0.8592

SWINGER pada metrik: relevance  (129 dialog)
           best_config worst_config    best   worst     gap
dialog_id                                                  
240        margin_part     finetune  0.9847  0.0203  0.9644
86       

## 8. Ekspor

In [8]:
summary.to_csv("per_metric_summary.csv")
profiles.to_csv("per_metric_profiles.csv")
agg.to_csv("per_metric_lowhigh_aggregate.csv")
cm.to_csv("cross_metric_correlation.csv")
print("Disimpan:")
print("  per_metric_summary.csv            (ICC & Spearman tiap metrik)")
print("  per_metric_profiles.csv           (profil tiap dialog x tiap metrik)")
print("  per_metric_lowhigh_aggregate.csv  (di berapa metrik tiap dialog always_low/high/swing)")
print("  cross_metric_correlation.csv      (korelasi antar metrik)")
print("""
RANGKUMAN CARA PAKAI:
- Tabel ringkasan (sel 3): metrik mana yang sifat bawaannya kuat (ICC tinggi) vs yang dikendalikan config.
- Korelasi antar metrik (sel 4): apakah 'dialog sulit' itu satu sifat umum atau berbeda tiap metrik.
- universal_low (sel 6): sampel sulit bawaan paling meyakinkan untuk error analysis.
- specific_low (sel 6): sampel yang gagal HANYA di metrik tertentu -> petunjuk kelemahan model yang spesifik.
- swinger per metrik (sel 7): kasus di mana satu config jelas menolong/merusak pada metrik tertentu.""")

Disimpan:
  per_metric_summary.csv            (ICC & Spearman tiap metrik)
  per_metric_profiles.csv           (profil tiap dialog x tiap metrik)
  per_metric_lowhigh_aggregate.csv  (di berapa metrik tiap dialog always_low/high/swing)
  cross_metric_correlation.csv      (korelasi antar metrik)

RANGKUMAN CARA PAKAI:
- Tabel ringkasan (sel 3): metrik mana yang sifat bawaannya kuat (ICC tinggi) vs yang dikendalikan config.
- Korelasi antar metrik (sel 4): apakah 'dialog sulit' itu satu sifat umum atau berbeda tiap metrik.
- universal_low (sel 6): sampel sulit bawaan paling meyakinkan untuk error analysis.
- specific_low (sel 6): sampel yang gagal HANYA di metrik tertentu -> petunjuk kelemahan model yang spesifik.
- swinger per metrik (sel 7): kasus di mana satu config jelas menolong/merusak pada metrik tertentu.
